# Compare code to assure same outputs

Comparing two ways to run the code to convince ourselves the outputs are the same and can be used as tests for further code updates.

1. Using Chao's original code on a downloaded hdf5 file
2. Using SlideRule to obtain the data and run the computations

Note: A generalized read-in function, along with a copy of the original function used by Chao to read in data, is located in the icepyx readdev.py file, imported locally below.

In [1]:
# not sure if this is needed anymore; restart kernel after running
%pip install --upgrade sliderule
%pip install icepyx



  Using cached sliderule-5.4.4-py3-none-any.whl.metadata (865 bytes)
Using cached sliderule-5.4.4-py3-none-any.whl (169 kB)
  Attempting uninstall: sliderule
    Found existing installation: sliderule 4.20.0
    Uninstalling sliderule-4.20.0:
      Successfully uninstalled sliderule-4.20.0
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
import glob

%load_ext autoreload
import icepyx as ipx
import aok as aok
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from datetime import datetime, timedelta
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
# import rasterio as rio
# from rasterio.session import AWSSession
from shapely.geometry import Polygon
import xarray as xr

import boto3
import earthaccess
import icepyx as ipx
from sliderule import sliderule, icesat2 #, io

In [7]:
# 2. Grab spatial inputs from test file

# TODO: add typing to these functions and the code that grabs the test cases, below.

import yaml
from pathlib import Path
from shapely.geometry import Point

def load_test_sites(path="./test_sites.yaml"):
    with Path(path).open("r") as f:
        return yaml.safe_load(f)

def get_region_by_name(name, sites=None):
    if sites is None:
        sites = load_test_sites()
    for site in sites["locations"]:
        if site["name"] == name:
            return site
    raise KeyError(f"Region not found: {name}")

def check_not_null(key):
    if key is None or all(l is None for l in key):
        return False
    else:
        return True

def get_bbox_shapely(lat, lon, buffer_deg) -> list:
    point = Point(lon, lat)
    # Creating a 'square' buffer
    bbox_poly = point.buffer(buffer_deg, cap_style=3) 
    return bbox_poly.bounds  # Returns (min_lon, min_lat, max_lon, max_lat)



In [8]:
site = get_region_by_name("cook_inlet_very_turbid_water")
#site = get_region_by_name("chesapeake_bay")
#site = get_region_by_name("rio_de_la_plata")



spatial = site["spatial_extent"]
if check_not_null(spatial["bbox"]):
    spatial_extent = spatial["bbox"]
elif check_not_null(spatial["latlon"]):
    spatial_extent = get_bbox_shapely(spatial["latlon"][0],
                                      spatial["latlon"][1],
                                      spatial["buffer"])
else:
    raise ValueError("Missing spatial extent")

if any([check_not_null(site["dates"]["start"]), check_not_null(site["dates"]["end"])]):
    
    temporal = [
        datetime.fromisoformat(site["dates"]["start"]),
        datetime.fromisoformat(site["dates"]["end"]),
    ]
else:
    raise ValueError("Missing temporal inputs")

## Calculate Kd from SlideRule

### Setting up configuration values

There are two ways to input configuration values:
1. from a `config.yaml` file
2. by manually entering them

For interactive coding, we recommend - and demonstrate below - a combination of these methods. This approach avoids the need to manually enter all inputs while providing flexibility to modify any inputs as desired.

:warning: Currently, the user MUST update the spatial and temporal extents included as defaults in the `config.yaml`, there or here.

In [9]:
# get configuration values from config.yaml
# supply `config_path='path\to\your\custom_config.yaml'` into `get_args()` if your file is not called `config.yaml`
kd_args = aok.main.get_args()

In [10]:
kd_args.spatial=spatial['bbox']

In [11]:
kd_args.temporal=[site["dates"]["start"], site["dates"]["end"]]

In [12]:
## Attempt to get the data using base...
from aok.core.datarequest import DataRequest

# Create a DataRequest object with spatial extent and temporal inputs
# TODO: add flags for other atl products
request = DataRequest(
    spatial=kd_args.spatial,
    date_range=kd_args.temporal,
    #(site["dates"]["start"], site["dates"]["end"]),
    output="geodataframe",
    beams= "all",
    download_dir="./test_data",
    need_atl03=True,
    need_atl24=False,  # Set to True if you want bathymetry data merged in
)

In [13]:
# Fetch data from SlideRule and get the geodataframe of photons
request.get_sliderule_data()

2026-07-16 14:28:37,270 - INFO - proxy request <AppServer.12444> querying resources for gebco


DataRequest(spatial=[-150.622053, 61.045382, -150.361757, 61.171146], date_range=['2025-08-21', '2025-08-22'], time_range=None, beams='all', output='geodataframe', download_dir='./test_data', need_atl03=True, need_atl24=False, need_gebco=True, need_shoreline=True, need_jpl_temperature=False, version_atl03=None, version_atl24=None, variables_atl03=None, variables_atl24=None, shoreline_data=None, land_ocean_mask=None, options={}, photons=                               segment_ph_cnt  ref_azimuth  gebco.fileid  \
time_ns                                                                    
2025-08-22 14:13:06.441814784               5     1.358289           [0]   
2025-08-22 14:13:06.442314752               5     1.358289           [0]   
2025-08-22 14:13:06.442314752               5     1.358289           [0]   
2025-08-22 14:13:06.442814720               5     1.358289           [0]   
2025-08-22 14:13:06.442814720               5     1.358289           [0]   
...                         

In [14]:
params = request.build_atl03_params()

In [15]:
request_atl03 = sliderule.run("atl03x", params)

2026-07-16 14:28:40,507 - INFO - proxy request <AppServer.12446> querying resources for gebco


In [16]:
request.photons.columns

Index(['segment_ph_cnt', 'ref_azimuth', 'gebco.fileid', 'segment_dist_x',
       'dist_ph_along', 'region', 'segment_length', 'gt',
       'spacecraft_velocity', 'solar_elevation', 'gebco.value',
       'high_rate/backg_c', 'reference_photon_lat', 'delta_time', 'h_ph',
       'rgt', 'ph_index', 'photon_height', 'background_rate', 'ref_elev',
       'beam_id', 'relative_AT_dist', 'y_atc', 'srcid', 'gebco.time_ns',
       'cycle', 'photon_conf', 'reference_photon_lon', 'background_rate_sr',
       'segment_id', 'ph_index_beg', 'quality_ph', 'geometry'],
      dtype='object')

In [17]:
request.photons

,segment_ph_cnt,ref_azimuth,gebco.fileid,segment_dist_x,dist_ph_along,region,segment_length,gt,spacecraft_velocity,solar_elevation,...,srcid,gebco.time_ns,cycle,photon_conf,reference_photon_lon,background_rate_sr,segment_id,ph_index_beg,quality_ph,geometry
time_ns,,,,,,,,,,,,,,,,,,,,,
2025-08-22 14:13:06.441814784,5,1.358289,[0],1.325918e+07,15.094666,5,20.021160,50,7098.117676,-2.454211,...,4,[2024-07-02T00:00:00.000000000],28,0,-150.361804,23622.046875,661992,1713836,0,POINT (-150.3618 61.12668)
2025-08-22 14:13:06.442314752,5,1.358289,[0],1.325918e+07,17.199505,5,20.021160,50,7098.117676,-2.454211,...,4,[2024-07-02T00:00:00.000000000],28,0,-150.361804,23622.046875,661992,1713836,0,POINT (-150.36173 61.12666)
2025-08-22 14:13:06.442314752,5,1.358289,[0],1.325918e+07,18.212816,5,20.021160,50,7098.117676,-2.454211,...,4,[2024-07-02T00:00:00.000000000],28,0,-150.361804,23622.046875,661992,1713836,0,POINT (-150.36179 61.12666)
2025-08-22 14:13:06.442814720,5,1.358289,[0],1.325918e+07,18.694416,5,20.021160,50,7098.117676,-2.454211,...,4,[2024-07-02T00:00:00.000000000],28,0,-150.361804,23622.046875,661992,1713836,0,POINT (-150.36162 61.12664)
2025-08-22 14:13:06.442814720,5,1.358289,[0],1.325918e+07,19.784868,5,20.021160,50,7098.117676,-2.454211,...,4,[2024-07-02T00:00:00.000000000],28,0,-150.361804,23622.046875,661992,1713836,0,POINT (-150.36168 61.12664)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-21 01:49:18.395253504,240,1.128917,[4294967296],6.817296e+06,19.385723,3,20.022305,40,7098.862793,26.565836,...,11,[2024-07-02T00:00:00.000000000],28,4,-150.503647,780746.937500,340059,6875061,0,POINT (-150.50366 61.17108)
2025-08-21 01:49:18.395253504,240,1.128917,[4294967296],6.817296e+06,19.386427,3,20.022305,40,7098.862793,26.565836,...,11,[2024-07-02T00:00:00.000000000],28,4,-150.503647,780746.937500,340059,6875061,0,POINT (-150.50366 61.17108)
2025-08-21 01:49:18.395253504,240,1.128917,[4294967296],6.817296e+06,19.386257,3,20.022305,40,7098.862793,26.565836,...,11,[2024-07-02T00:00:00.000000000],28,4,-150.503647,780746.937500,340059,6875061,0,POINT (-150.50366 61.17108)


In [18]:
type(request.photons)

geopandas.geodataframe.GeoDataFrame

In [19]:
sr_photons = request.photons

In [20]:
# Display the result
print(f"Retrieved {len(sr_photons)} photons from SlideRule")
sr_photons.head()

Retrieved 137857 photons from SlideRule


,segment_ph_cnt,ref_azimuth,gebco.fileid,segment_dist_x,dist_ph_along,region,segment_length,gt,spacecraft_velocity,solar_elevation,...,srcid,gebco.time_ns,cycle,photon_conf,reference_photon_lon,background_rate_sr,segment_id,ph_index_beg,quality_ph,geometry
time_ns,,,,,,,,,,,,,,,,,,,,,
2025-08-22 14:13:06.441814784,5,1.358289,[0],1.325918e+07,15.094666,5,20.02116,50,7098.117676,-2.454211,...,4,[2024-07-02T00:00:00.000000000],28,0,-150.361804,23622.046875,661992,1713836,0,POINT (-150.3618 61.12668)
2025-08-22 14:13:06.442314752,5,1.358289,[0],1.325918e+07,17.199505,5,20.02116,50,7098.117676,-2.454211,...,4,[2024-07-02T00:00:00.000000000],28,0,-150.361804,23622.046875,661992,1713836,0,POINT (-150.36173 61.12666)
2025-08-22 14:13:06.442314752,5,1.358289,[0],1.325918e+07,18.212816,5,20.02116,50,7098.117676,-2.454211,...,4,[2024-07-02T00:00:00.000000000],28,0,-150.361804,23622.046875,661992,1713836,0,POINT (-150.36179 61.12666)
2025-08-22 14:13:06.442814720,5,1.358289,[0],1.325918e+07,18.694416,5,20.02116,50,7098.117676,-2.454211,...,4,[2024-07-02T00:00:00.000000000],28,0,-150.361804,23622.046875,661992,1713836,0,POINT (-150.36162 61.12664)
2025-08-22 14:13:06.442814720,5,1.358289,[0],1.325918e+07,19.784868,5,20.02116,50,7098.117676,-2.454211,...,4,[2024-07-02T00:00:00.000000000],28,0,-150.361804,23622.046875,661992,1713836,0,POINT (-150.36168 61.12664)


Expected Tracks: 1214, 1215
Expected Cycles: 06

## Compare Outputs



In [22]:
aok.main.run_pipeline(kd_args)

2026-07-16 14:43:17,092 - INFO - proxy request <AppServer.13883> querying resources for gebco
2026-07-16 14:43:17,093 - INFO - request <AppServer.13883> retrieved 2 resources
2026-07-16 14:43:17,094 - INFO - Starting proxy for atl03x to process 2 resource(s) with 2 thread(s)
2026-07-16 14:43:17,094 - INFO - Attempt 1 of 3 ATL09 CMR request: {"name_filter":"*_103628??_*","asset":"icesat2-atl09"}
2026-07-16 14:43:17,095 - INFO - Attempt 1 of 3 ATL09 CMR request: {"asset":"icesat2-atl09","name_filter":"*_101328??_*"}
2026-07-16 14:43:19,550 - INFO - Processing atmospheric data from ATL09_20250821013321_10132801_007_01.h5
2026-07-16 14:43:19,550 - INFO - Processing atmospheric data from ATL09_20250822134200_10362801_007_01.h5
2026-07-16 14:43:20,150 - INFO - request <AppServer.13496> on ATL03_20250822140806_10362805_007_01.h5 generated dataframe [gt1r] with 0 rows and 16 columns
2026-07-16 14:43:21,439 - INFO - request <AppServer.13496> on ATL03_20250822140806_10362805_007_01.h5 generated 

Processing binning for beam: 1


KeyError: 'lat'

In [ ]:
#kd_df = pd.read_csv("./results/2023-07-04_subsurface_photons.csv")
kd_df = pd.read_csv("./results/2025-08-21_subsurface_photons.csv")

In [ ]:
kd_df.columns

dictionary location:
orbit_info/sc orient : direction of spacecraft; backward (0), forward (1), transition (2)
ancillary_data/atlas_engineering/transmit/tx_pulse_energy mean, sd, min, max of transmit energy; strong transmit energy = strong beam and vice versa.

spot - odds are strong, evens are weak; independent of spacecraft orientation
